# Feature selection and Optuna tuning

Run from top to bottom. Quick mode is a patient subset, not final results. Notebook outputs are separate from production reports.

In [ ]:
from pathlib import Path
import sys, os
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src'/'features.py').exists()), None)
assert ROOT is not None, 'Open the notebook from the project repository'
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.notebook_utils import *
QUICK = True  # False uses the entire eligible dataset; keep consistent across notebooks
OUT = ROOT/'reports'/('notebook_quick' if QUICK else 'notebook_full')
OUT.mkdir(parents=True, exist_ok=True)


Load cohort

In [ ]:
raw, x, y, g, parts = load_cohort(ROOT, QUICK)
train, cal, tune, test = (parts[k] for k in ("train","calibration","threshold","test"))
X, Y, G = x.iloc[train], y.iloc[train], g.iloc[train]
display(split_table(y,g,parts))

Rank features using development-fold permutation importance; the subsequent CV scores are exploratory.

In [ ]:
from src.advanced_experiment import expanded,pipeline
from sklearn.inspection import permutation_importance
z,_=expanded(raw); A=z.iloc[train]; ranks=[]
for t,v in folds(A,Y,G):
    m=pipeline(A,QUICK).fit(A.iloc[t],Y.iloc[t]); v=v[:min(300,len(v))]; ranks.append(permutation_importance(m,A.iloc[v],Y.iloc[v],scoring='average_precision',n_repeats=1,random_state=42).importances_mean)
ranked=pd.Series(np.mean(ranks,axis=0),index=A.columns).sort_values(ascending=False); display(ranked.head(20)); cols=list(ranked.head(20).index)

Tune XGBoost using Optuna and grouped cross-validation

In [ ]:
import optuna
def objective(trial):
    m=pipeline(A[cols],QUICK).set_params(model__max_depth=trial.suggest_int('max_depth',2,6),model__learning_rate=trial.suggest_float('learning_rate',.01,.2,log=True),model__n_estimators=trial.suggest_int('n_estimators',20,60 if QUICK else 700),model__scale_pos_weight=trial.suggest_float('scale_pos_weight',1,10,log=True),model__reg_lambda=trial.suggest_float('reg_lambda',.1,20,log=True))
    _,r=group_cv(m,A[cols],Y,G); return r[:,0].mean()
study=optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler(seed=42)); study.optimize(objective,n_trials=2 if QUICK else 50)
print(study.best_params); d=study.trials_dataframe(); display(d); d.value.cummax().plot(marker='o',title='Best CV AP so far'); plt.show(); d.to_csv(OUT/'optuna.csv',index=False)